# Physics-informed fine-tuning of the PhysicsNeMo Stokes surrogate on a Kaggle GPU

Launcher for [physicsnemo-stokes-finetuning](https://github.com/AdebanjiAdelowo/physicsnemo-stokes-finetuning). It checks out one commit of the repository and calls its scripts; it contains no model or training code.

Session settings: **Accelerator** an NVIDIA GPU, **Internet** on.

| Step | What runs | When |
|---|---|---|
| A | install, dataset download with SHA-256 check, seeded split, tests | always |
| B | CUDA smoke study (both stages, tiny) | always |
| C | timing and memory probe: one epoch of the data-driven stage, 200 fine-tuning iterations | always |
| D | full study, `configs/full.yaml`: 500 epochs on 500 meshes, then 10000 fine-tuning iterations on each of 10 test meshes | `RUN_FULL = True` |

Before D the notebook projects the runtime from the probe and compares it with what is left of `SESSION_BUDGET_HOURS`. If the projection does not fit, the step is **not started and nothing is reduced**; the notebook reports the numbers and finishes.

There is no separate `RUN_OFFICIAL` step. `configs/official.yaml` holds NVIDIA's settings and fine-tunes test mesh 7 only, as the upstream script does; `configs/full.yaml` uses the same settings and seed for both stages and fine-tunes test meshes 0 to 9, so the upstream case is sample 7 of step D.

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-stokes-finetuning.git"
REF = "SET_TO_FULL_40_CHARACTER_COMMIT_SHA"   # output of `git rev-parse HEAD` for the commit to run
RUN_FULL = False                              # step D: only the value True starts it
SESSION_BUDGET_HOURS = 9.0                    # GPU time you are willing to spend in this session; check Kaggle's current limits
GITHUB_TOKEN_SECRET = None                    # name of a Kaggle secret holding a GitHub token; only for a private repository

import re, time
T0 = time.time()
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit SHA, not a branch or tag."
WORK = "/kaggle/working"
REPO = f"{WORK}/physicsnemo-stokes-finetuning"

def hours_left():
    return SESSION_BUDGET_HOURS - (time.time() - T0) / 3600

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "CUDA is not available. Select a GPU accelerator; this notebook does not fall back to CPU."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
import os, subprocess
def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

def tree_is_clean():
    return not subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip()

if not os.path.isdir(REPO):
    url = REPO_URL
    if GITHUB_TOKEN_SECRET:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET) + "@")
    subprocess.run(["git", "clone", "--quiet", url, REPO], check=True)   # not echoed: the URL may hold a token
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
assert head == REF, f"checked out {head}, expected {REF}"
assert tree_is_clean(), "the working tree is not clean"
print("commit", head)

## Step A: environment, dataset, tests

The requirements pin PhysicsNeMo to a commit of its main branch, which needs `torch>=2.13`; `pip` may replace the PyTorch build of the image. `torch_scatter` has no prebuilt wheel for that PyTorch and is compiled from source (CPU sources only; the sum aggregation of MeshGraphNet runs through `Tensor.scatter_add_` on the GPU). The versions are printed afterwards in a new process, and CUDA is checked again.

In [ ]:
sh("pip install -q -r requirements.txt", cwd=REPO)
sh("FORCE_ONLY_CPU=1 pip install -q --no-build-isolation --no-binary torch-scatter torch-scatter==2.1.2", cwd=REPO)
sh('python -c "import torch, physicsnemo, torch_geometric, torch_scatter; assert torch.cuda.is_available(); '
   "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| physicsnemo', physicsnemo.__version__, "
   "'| torch_geometric', torch_geometric.__version__, '| torch_scatter', torch_scatter.__version__, '|', torch.cuda.get_device_name(0))\"", cwd=REPO)

In [ ]:
sh("python scripts/prepare_data.py", cwd=REPO)      # 172 MB from NGC, SHA-256 verified, 800/100/100 split with seed 0
assert tree_is_clean(), "preparing the data modified the clone"
sh("python -m pytest -q", cwd=REPO)

## Step B: CUDA smoke study

Six training meshes, two epochs, twenty fine-tuning iterations on two test meshes. The numbers are not results.

In [ ]:
sh("python scripts/run_study.py --config smoke device=cuda name=smoke_cuda", cwd=REPO)
sh("python scripts/plot_results.py --study runs/smoke_cuda --out runs/smoke_cuda/figures", cwd=REPO)

## Step C: timing and memory probe

One epoch of the data-driven stage of `configs/full.yaml` (500 meshes) and 200 fine-tuning iterations on test mesh 7. The projection scales these two measurements to 500 epochs and to 10 meshes of 10000 iterations. It is an extrapolation and includes start-up cost, so it tends to overestimate.

In [ ]:
import json
sh('python scripts/run_study.py --config full name=full_probe epochs=1 pi_iters=200 "finetune.samples=[7]" '
   '"evaluation.plot_samples=[7]" training.checkpoint_every=1', cwd=REPO)
train = json.load(open(f"{REPO}/runs/full_probe/train_metrics.json"))
tune = json.load(open(f"{REPO}/runs/full_probe/finetune/finetune_7.json"))
meta = json.load(open(f"{REPO}/runs/full_probe/study_metadata.json"))["sessions"][-1]
train_h = train["seconds_per_iteration"] * 500 * 500 / 3600      # 500 epochs x 500 meshes in configs/full.yaml
tune_h = tune["seconds_per_iteration"] * 10000 * 10 / 3600       # 10000 iterations x 10 test meshes in configs/full.yaml
full_projection_h = train_h + tune_h
print(f"GPU: {meta['gpu_name']} | torch {meta['torch_version']} | CUDA {meta['cuda_version']}")
print(f"data-driven stage: {train['seconds_per_iteration'] * 1e3:.1f} ms per iteration, peak {train['peak_cuda_memory_mb']:.0f} MB -> {train_h:.2f} h for 250000 iterations")
print(f"fine-tuning stage: {tune['seconds_per_iteration'] * 1e3:.1f} ms per iteration, peak {tune['peak_cuda_memory_mb']:.0f} MB -> {tune_h:.2f} h for 10 meshes")
print(f"PROJECTION for step D: {full_projection_h:.1f} h. Left of the session budget: {hours_left():.1f} h.")

## Step D: full study (`configs/full.yaml`)

The MeshGraphNet checkpoint before the physics-informed stage and the ten fine-tuning networks after it are kept in `runs/full/checkpoints` and stored in the checkpoint archive. Both are evaluated on the same ten test meshes.

In [ ]:
full_done = False
if RUN_FULL is not True:
    print("Step D skipped: RUN_FULL is False.")
elif full_projection_h > hours_left():
    print(f"STEP D NOT STARTED: projected {full_projection_h:.1f} h, {hours_left():.1f} h left of SESSION_BUDGET_HOURS = {SESSION_BUDGET_HOURS}.")
    print("The configuration was not reduced. Decide how to proceed before changing configs/full.yaml.")
else:
    sh("python scripts/run_study.py --config full --resume", cwd=REPO)
    sh("python scripts/plot_results.py --study runs/full --out runs/full/figures", cwd=REPO)
    sh("python scripts/results.py package runs/full", cwd=REPO)
    sh("python scripts/results.py bundle runs/full", cwd=REPO)
    sh(f"cp runs/full.zip {WORK}/physicsnemo-stokes-finetuning-full.zip && cp runs/full_checkpoints.zip {WORK}/physicsnemo-stokes-finetuning-full-checkpoints.zip", cwd=REPO)
    sh(f"cd runs/full && zip -q -r {WORK}/physicsnemo-stokes-finetuning-full-figures.zip figures", cwd=REPO)
    full_done = True
    print("STEP D COMPLETE.")

In [ ]:
import glob
print("Archives in /kaggle/working:")
for path in sorted(glob.glob(f"{WORK}/physicsnemo-stokes-finetuning-*.zip")):
    print(f"  {path}  {os.path.getsize(path) / 2**20:.1f} MB")
print(f"Elapsed: {(time.time() - T0) / 3600:.2f} h. Steps A to C: done. Step D: {'done' if full_done else 'not run'}.")
if not full_done:
    print("KAGGLE CUDA CHECKS COMPLETE. No full study was run.")